In [ ]:
#!pip install qnexus

In [ ]:
import qnexus as qnx
from pytket import Circuit
from datetime import datetime

# 0. ローカル環境からqnexusを利用したNexusアクセス設定

## ID、パスワードを入力してNexusにアクセス

In [ ]:
#qnx.login()
#qnx.login_with_credentials()

# 1. Nexusにあるprojects, teamsを確認

In [ ]:
all_my_projects = qnx.projects.get_all()

In [ ]:
all_my_projects.df()

In [ ]:
all_my_projects.count()

In [ ]:
all_my_teams = qnx.teams.get_all()

In [ ]:
all_my_teams.df()

# 2. NexusのprojectsでJobを管理する

## a. TKETで量子回路を作成する

In [ ]:
from pytket import Circuit

In [ ]:
circ = Circuit(2)
circ.H(0).CX(0,1)
circ.measure_all()

## b. Nexusのprojectsを指定/作成し、アクティベーションする

In [ ]:
my_project_ref = qnx.projects.get_or_create("My Project")# input your project
qnx.context.set_active_project(my_project_ref)

## c. projectsにTKET量子回路を保存し、量子回路を最適化し、実機/エミュレータで実行する

### 作成した量子回路をprojectsに保存

In [ ]:
my_circ = qnx.circuits.upload(name="my_circ", circuit=circ)

In [ ]:
my_circ.download_circuit()

In [ ]:
from pytket.circuit.display import render_circuit_jupyter
render_circuit_jupyter(my_circ.download_circuit())

### 作成した量子回路を量子デバイスで実行するために最適化

In [ ]:
device_name = "H2-Emulator"
my_quantinuum_config = qnx.QuantinuumConfig(
    device_name=device_name,
)

In [ ]:
my_comp = qnx.start_compile_job(
    name = f"my_circ compilation {datetime.now()}",
    programs=[my_circ],#複数の量子回路をコンパイルすることが可能 ex. programs=[my_circ1,my_circ2]
    backend_config=my_quantinuum_config,
    optimisation_level = 3,
)

In [ ]:
my_comp.df()

In [ ]:
qnx.jobs.status(my_comp)

In [ ]:
qnx.jobs.wait_for(my_comp)

In [ ]:
comp_result = qnx.jobs.results(my_comp)[0].get_output()

In [ ]:
comp_circ = comp_result.download_circuit()

In [ ]:
comp_circ

In [ ]:
render_circuit_jupyter(comp_circ)

### 最適化した量子回路を実機/エミュレータに実行

In [ ]:
my_exe = qnx.start_execute_job(
    name = f"my_circ execute {datetime.now()}",
    programs=[comp_result],
    n_shots=[10],#複数の量子回路をコンパイルしている場合には、各回路のショットリストを渡す
#    backend_config=qnx.QuantinuumConfig(device_name="H2-1LE")
    backend_config=qnx.QuantinuumConfig(device_name="H2-Emulator")
)

In [ ]:
qnx.jobs.status(my_exe)

In [ ]:
qnx.jobs.wait_for(my_exe)

In [ ]:
result = qnx.jobs.results(my_exe)

In [ ]:
result[0].download_result().get_counts()

In [ ]:
qnx.jobs.get_all().df()

## d. Nexusデータベースの過去のジョブを抽出する

In [ ]:
my_project_ref = qnx.projects.get_or_create("My Project")
qnx.context.set_active_project(my_project_ref)

In [ ]:
qnx.jobs.get_all().df()

#### NexusデータベースからJob（コンパイルジョブ）を抽出

In [ ]:
jobref = qnx.jobs.get(id="5eef6b8f-893e-4f6e-8303-93dd69d1c1f2")
jobref

In [ ]:
jobcomp = qnx.jobs.results(jobref)[0]
jobcomp

In [ ]:
jobcomp.get_input().download_circuit()

In [ ]:
jobcomp.get_output().download_circuit()

#### NexusデータベースからJob(実行ジョブ)を抽出

In [ ]:
jobref = qnx.jobs.get(id="be4b0f92-cd66-4f89-9442-62e05cdc6432")
jobref

In [ ]:
jobres = qnx.jobs.results(jobref)[0]
jobres

In [ ]:
jobres.get_input().download_circuit()

In [ ]:
jobres.download_result().get_counts()

# Syntax checkerの利用 H2利用の前には必ず実行ください。  
## (Jobが問題なく実行可能か、HQCコストなどの確認)

In [ ]:
device_name = "H2-1SC"

config = qnx.QuantinuumConfig(device_name=device_name)

job_name = f"execution-job-qir-{datetime.now()}"
ref_execute_job = qnx.start_execute_job(
    programs=[comp_result],
    n_shots=[1000],
    backend_config=config,
    name=job_name,
)

qnx.jobs.wait_for(ref_execute_job)

## HQCの算出だけであれば、関数`cost`で可能

In [ ]:
qnx.client.circuits.cost(comp_result,n_shots=1000,backend_config=config)